# Delta Ladder (Risk Ladder) for 20Y Pay-Fixed 4.50% SOFR Swap
## Methodology

For each tenor in your curve:
1. Calculate base NPV using the base curve
2. Bump the par rate by 1 basis point (0.01%)
3. Re-bootstrap the curve with the bumped rate
4. Revalue the swap
5. **Delta = NPV_bumped - NPV_base**

This gives you the dollar sensitivity per 1bp move in each input rate.

---

## 1. Import Libraries

In [64]:
import math
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) > 0.01 else f'{x:.6f}')

## 2. Configuration & Data

In [66]:
# SWAP SPECIFICATION

VALUATION_DATE = dt.date(2024, 12, 31)
SWAP_MATURITY_YEARS = 20
FIXED_RATE = 0.045  # 4.50%
NOTIONAL = 10_000_000  # $10MM

# PAR RATES DATA
# These are representative USD SOFR swap par rates as of Dec 31, 2024
# Replace with your actual data from Excel files

par_rates_data = pd.DataFrame({
    'Term': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20],
    'Unit': ['Y']*13,
    'mid_rate_pct': [4.175870, 4.077600, 4.054550, 4.041500, 4.038500, 4.043500, 4.049050, 
                     4.055355, 4.062395, 4.071001, 4.094000, 4.118880, 4.101500]
})

print(f"Valuation Date: {VALUATION_DATE}")
print(f"Swap: {SWAP_MATURITY_YEARS}Y Pay-Fixed {FIXED_RATE*100:.2f}%")
print(f"Notional: ${NOTIONAL:,}")
print(f"\nPar Rates Loaded: {len(par_rates_data)} tenors")
display(par_rates_data)

Valuation Date: 2024-12-31
Swap: 20Y Pay-Fixed 4.50%
Notional: $10,000,000

Par Rates Loaded: 13 tenors


,Term,Unit,mid_rate_pct
0,1,Y,4.18
1,2,Y,4.08
2,3,Y,4.05
3,4,Y,4.04
4,5,Y,4.04
5,6,Y,4.04
6,7,Y,4.05
7,8,Y,4.06
8,9,Y,4.06
9,10,Y,4.07


## 3. Helper Functions

In [68]:
def add_tenor(start: dt.date, term: int, unit: str) -> dt.date:
    """Add a tenor to a start date."""
    unit = str(unit).upper().strip()
    if unit in ("YR", "Y", "YRS", "YEAR", "YEARS"):
        return start + relativedelta(years=int(term))
    elif unit in ("MO", "M", "MOS", "MONTH", "MONTHS"):
        return start + relativedelta(months=int(term))
    elif unit in ("WK", "W", "WKS", "WEEK", "WEEKS"):
        return start + dt.timedelta(days=7*int(term))
    raise ValueError(f"Unknown unit: {unit}")

def yearfrac_act360(d0: dt.date, d1: dt.date) -> float:
    """Calculate year fraction using ACT/360."""
    return (d1 - d0).days / 360.0

def fixed_pay_dates_annual(start: dt.date, end: dt.date):
    """Generate annual payment dates."""
    dates = []
    cur = start + relativedelta(years=1)
    while cur < end:
        dates.append(cur)
        cur = cur + relativedelta(years=1)
    dates.append(end)
    return dates

## 4. Discount Curve Class

In [70]:
class DiscountCurve:
    """Simple discount curve with log-linear interpolation."""
    
    def __init__(self, val_date: dt.date):
        self.val_date = val_date
        self.nodes = {val_date: 1.0}
    
    def add(self, date: dt.date, df: float):
        """Add a discount factor node."""
        self.nodes[date] = float(df)
    
    def df(self, date: dt.date) -> float:
        """Get discount factor at date (with log-linear interpolation)."""
        if date in self.nodes:
            return self.nodes[date]
        
        dates = sorted(self.nodes.keys())
        if date <= dates[0]:
            return self.nodes[dates[0]]
        if date >= dates[-1]:
            return self.nodes[dates[-1]]
        
        # Find surrounding nodes
        for i in range(1, len(dates)):
            if dates[i] > date:
                dL, dR = dates[i-1], dates[i]
                break
        
        # Log-linear interpolation on discount factors
        tL = (dL - self.val_date).days / 365.0
        tR = (dR - self.val_date).days / 365.0
        t = (date - self.val_date).days / 365.0
        w = (t - tL) / (tR - tL)
        ln_df = (1-w) * math.log(self.nodes[dL]) + w * math.log(self.nodes[dR])
        return math.exp(ln_df)
    
    def node_table(self) -> pd.DataFrame:
        """Return table of curve nodes."""
        return pd.DataFrame([
            {"date": d, "df": self.nodes[d]} 
            for d in sorted(self.nodes.keys())
        ])

## 5. Bootstrap Function

In [72]:
def bootstrap_from_swaps(swap_data: pd.DataFrame, val_date: dt.date, max_years: int = 30):
    """
    Bootstrap discount curve from annual-pay swap par rates.
    
    Uses the standard iterative bootstrap:
    For a par swap: PV_fixed = PV_float
    => rate * sum(tau_i * DF_i) + rate * tau_N * DF_N = 1 - DF_N
    => DF_N = (1 - rate * A) / (1 + rate * tau_N)
    
    where A = sum of (tau_i * DF_i) for all known periods
    """
    curve = DiscountCurve(val_date)
    
    cutoff = val_date + relativedelta(years=max_years)
    
    # Add maturity dates and sort
    data = swap_data.copy()
    data["maturity"] = data.apply(
        lambda r: add_tenor(val_date, int(r["Term"]), r["Unit"]), 
        axis=1
    )
    data = data[data["maturity"] <= cutoff].sort_values("maturity").reset_index(drop=True)
    
    # Bootstrap each tenor
    for _, r in data.iterrows():
        end = r["maturity"]
        rate = float(r["mid_rate_pct"]) / 100.0  # Convert from % to decimal
        
        pay_dates = fixed_pay_dates_annual(val_date, end)
        
        # A = sum of (tau_i * DF_i) for all periods before the last
        A = 0.0
        prev = val_date
        for d in pay_dates[:-1]:
            tau = yearfrac_act360(prev, d)
            A += tau * curve.df(d)
            prev = d
        
        # Solve for DF at maturity
        tauN = yearfrac_act360(prev, pay_dates[-1])
        dfN = (1.0 - rate * A) / (1.0 + rate * tauN)
        
        curve.add(pay_dates[-1], dfN)
    
    return curve

## 6. Swap Valuation Function

In [74]:
def price_pay_fixed_swap(curve: DiscountCurve, 
                        start: dt.date, 
                        end: dt.date, 
                        fixed_rate: float, 
                        notional: float = 1.0):
    """
    Price a pay-fixed receive-floating swap.
    
    Returns: (pv_fixed, pv_float, npv)
    where NPV is from payer perspective (receive_float - pay_fixed)
    """
    pay_dates = fixed_pay_dates_annual(start, end)
    
    pv_fixed = 0.0
    pv_float = 0.0
    
    prev = start
    for d in pay_dates:
        tau = yearfrac_act360(prev, d)
        
        df_prev = curve.df(prev)
        df_d = curve.df(d)
        
        # Simple forward rate from discount factors
        fwd = (df_prev / df_d - 1.0) / tau
        
        # Fixed leg cashflow
        fixed_cf = notional * fixed_rate * tau
        pvF = fixed_cf * df_d
        pv_fixed += pvF
        
        # Floating leg cashflow
        float_cf = notional * fwd * tau
        pvL = float_cf * df_d
        pv_float += pvL
        
        prev = d
    
    # NPV from payer perspective
    npv = pv_float - pv_fixed
    
    return pv_fixed, pv_float, npv


## 7. Base Swap Valuation

In [76]:
# Bootstrap base curve
print("Bootstrapping base curve...")
base_curve = bootstrap_from_swaps(par_rates_data, VALUATION_DATE)

# Value the swap
swap_start = VALUATION_DATE
swap_end = VALUATION_DATE + relativedelta(years=SWAP_MATURITY_YEARS)

pv_fixed_base, pv_float_base, npv_base = price_pay_fixed_swap(
    base_curve, swap_start, swap_end, FIXED_RATE, NOTIONAL
)

print("BASE SWAP VALUATION")
print(f"PV Fixed Leg:     ${pv_fixed_base:15,.2f}")
print(f"PV Floating Leg:  ${pv_float_base:15,.2f}")
print(f"NPV (Payer):      ${npv_base:15,.2f}")


Bootstrapping base curve...
BASE SWAP VALUATION
PV Fixed Leg:     $   6,103,700.03
PV Floating Leg:  $   5,651,974.59
NPV (Payer):      $    -451,725.44


In [77]:
# Display results
print("DELTA LADDER RESULTS")
print()

# Style the dataframe
styled_ladder = delta_ladder.style\
    .format({
        'Par Rate (%)': '{:.2f}',
        'Bumped Rate (%)': '{:.2f}',
        'Delta (PV01)': '${:,.2f}'
    })\
    .background_gradient(subset=['Delta (PV01)'], cmap='RdYlGn', vmin=-5000, vmax=20000)

display(styled_ladder)

# Summary statistics
print("SUMMARY STATISTICS")
print(f"Base NPV:                ${npv:,.2f}")
print(f"Total PV01 (sum):        ${delta_ladder['Delta (PV01)'].sum():,.2f}")
print(f"Max Positive Delta:      ${delta_ladder['Delta (PV01)'].max():,.2f} at {delta_ladder.loc[delta_ladder['Delta (PV01)'].idxmax(), 'Tenor']}")
print(f"Max Negative Delta:      ${delta_ladder['Delta (PV01)'].min():,.2f} at {delta_ladder.loc[delta_ladder['Delta (PV01)'].idxmin(), 'Tenor']}")
print(f"Number of instruments:   {len(delta_ladder)}")

DELTA LADDER RESULTS



,Tenor,Par Rate (%),Bumped Rate (%),Delta (PV01)
0,1Y,4.18,4.19,$2.13
1,2Y,4.08,4.09,$4.35
2,3Y,4.05,4.06,$6.67
3,4Y,4.04,4.05,$9.11
4,5Y,4.04,4.05,$11.60
5,6Y,4.04,4.05,$14.21
6,7Y,4.05,4.06,$16.93
7,8Y,4.06,4.07,$19.81
8,9Y,4.06,4.07,$22.71
9,10Y,4.07,4.08,$36.52


SUMMARY STATISTICS
Base NPV:                $-451,725.44
Total PV01 (sum):        $14,349.19
Max Positive Delta:      $15,148.50 at 20Y
Max Negative Delta:      $-1,068.79 at 15Y
Number of instruments:   13
